In [ ]:
#Sterownik rozmyty Warszawska-Francuska

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

def symulacja_skrzyzowania_warszawska_francuska(input_kolejka, input_czas):
    """
    Symuluje decyzję sterownika dla skrzyżowania w Katowicach.

    Args:
        input_kolejka (int): Liczba samochodów w kolejce (0-30).
        input_czas (int): Czas oczekiwania w sekundach (0-120).
    """

    # 1. Definiowanie zmiennych lingwistycznych (Antecedents & Consequent)
    # Kolejka na ul. Francuskiej (zakładamy zakres 0-30 aut na pasie detekcji)
    kolejka = ctrl.Antecedent(np.arange(0, 31, 1), 'kolejka')

    # Czas oczekiwania na zmianę świateł (0-120 sekund)
    czas_oczekiwania = ctrl.Antecedent(np.arange(0, 121, 1), 'czas_oczekiwania')

    # Wyjście: Przedłużenie światła zielonego (0-60 sekund)
    przedluzenie = ctrl.Consequent(np.arange(0, 61, 1), 'przedluzenie')

    # 2. Definiowanie funkcji przynależności (Membership Functions)
    # Używamy kształtów trójkątnych (trimf) dla prostoty obliczeń

    # Dla Kolejki
    kolejka['mala'] = fuzz.trimf(kolejka.universe, [0, 0, 10])
    kolejka['srednia'] = fuzz.trimf(kolejka.universe, [5, 15, 25])
    kolejka['duza'] = fuzz.trimf(kolejka.universe, [20, 30, 30])

    # Dla Czasu Oczekiwania (im dłużej czekają inni, tym mniejsza szansa na przedłużenie obecnego)
    czas_oczekiwania['krotki'] = fuzz.trimf(czas_oczekiwania.universe, [0, 0, 40])
    czas_oczekiwania['sredni'] = fuzz.trimf(czas_oczekiwania.universe, [20, 60, 100])
    czas_oczekiwania['dlugi'] = fuzz.trimf(czas_oczekiwania.universe, [80, 120, 120])

    # Dla Przedłużenia Zielonego
    przedluzenie['brak'] = fuzz.trimf(przedluzenie.universe, [0, 0, 10])
    przedluzenie['krotkie'] = fuzz.trimf(przedluzenie.universe, [5, 20, 35])
    przedluzenie['dlugie'] = fuzz.trimf(przedluzenie.universe, [25, 60, 60])

    # 3. Baza Reguł (Rule Base) - Logika sterownika
    # Tutaj definiujemy "inteligencję" skrzyżowania

    rule1 = ctrl.Rule(kolejka['mala'], przedluzenie['brak'])
    rule2 = ctrl.Rule(kolejka['srednia'] & czas_oczekiwania['krotki'], przedluzenie['krotkie'])
    # Jeśli kolejka jest duża, a inni czekają krótko -> daj długie zielone (priorytet rozładowania Francuskiej)
    rule3 = ctrl.Rule(kolejka['duza'] & czas_oczekiwania['krotki'], przedluzenie['dlugie'])
    # Jeśli kolejka jest duża, ale inni czekają już bardzo długo -> skróć przedłużenie (sprawiedliwość)
    rule4 = ctrl.Rule(kolejka['duza'] & czas_oczekiwania['dlugi'], przedluzenie['krotkie'])

    # Tworzenie systemu sterowania
    system_sterowania = ctrl.ControlSystem([rule1, rule2, rule3, rule4])
    symulacja = ctrl.ControlSystemSimulation(system_sterowania)

    # 4. Wprowadzenie danych wejściowych
    symulacja.input['kolejka'] = input_kolejka
    symulacja.input['czas_oczekiwania'] = input_czas

    # 5. Obliczenia (wnioskowanie + defuzyfikacja)
    try:
        symulacja.compute()
        wynik = symulacja.output['przedluzenie']
        return round(wynik, 2)
    except:
        return 0

# --- Przykładowe scenariusze dla Katowic ---

# Scenariusz A: Szczyt popołudniowy na Francuskiej (dużo aut),
# poprzeczna Warszawska dopiero dostała czerwone (krótki czas oczekiwania).
scenariusz_a = symulacja_skrzyzowania_warszawska_francuska(input_kolejka=28, input_czas=10)

# Scenariusz B: Noc, pojedyncze auto na Francuskiej.
scenariusz_b = symulacja_skrzyzowania_warszawska_francuska(input_kolejka=2, input_czas=10)

# Scenariusz C: Duży korek na Francuskiej, ale na Warszawskiej tramwaj czeka już długo (długi czas oczekiwania).
scenariusz_c = symulacja_skrzyzowania_warszawska_francuska(input_kolejka=28, input_czas=90)

print(f"Scenariusz A (Szczyt): Przedłuż zielone o {scenariusz_a} s")
print(f"Scenariusz B (Pusto): Przedłuż zielone o {scenariusz_b} s")
print(f"Scenariusz C (Konflikt): Przedłuż zielone o {scenariusz_c} s")

Scenariusz A (Szczyt): Przedłuż zielone o 47.75 s
Scenariusz B (Pusto): Przedłuż zielone o 3.44 s
Scenariusz C (Konflikt): Przedłuż zielone o 20.0 s
